# EDA: TalkingData AdTracking

Mechanical descriptive EDA driven by `openbotrisk.eda`.

In [1]:
import sys, time, datetime as dt
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

from openbotrisk.eda.loaders import load_talkingdata_meta
from openbotrisk.eda.descriptive import label_balance
from openbotrisk.eda.report import write_report

DATA_DIR = REPO_ROOT / 'data' / 'talkingdata-adtracking-fraud-detection'
REPORT_PATH = REPO_ROOT / 'working' / 'eda' / 'talkingdata-adtracking.md'

t0 = time.time()
meta = load_talkingdata_meta(DATA_DIR)
elapsed = time.time() - t0
print(f'meta in {elapsed:.1f}s; rows={meta["row_count"]:,}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

meta in 6.2s; rows=184,903,890


In [2]:
schema = meta['schema']
nulls = meta['null_table']
card = meta['cardinality']
lbl = meta['label_balance']
tstats = meta['time_stats']
sample = meta['sample']
schema

,column_name,column_type,null,key,default,extra
0,ip,BIGINT,YES,None,None,None
1,app,BIGINT,YES,None,None,None
2,device,BIGINT,YES,None,None,None
3,os,BIGINT,YES,None,None,None
4,channel,BIGINT,YES,None,None,None
5,click_time,TIMESTAMP,YES,None,None,None
6,attributed_time,TIMESTAMP,YES,None,None,None
7,is_attributed,BIGINT,YES,None,None,None


In [3]:
def fmt_size(b):
    for u in ['B','KB','MB','GB']:
        if b < 1024: return f'{b:.1f} {u}'
        b /= 1024
    return f'{b:.1f} TB'

file_lines = '\n'.join(
    f'- `{name}` — {fmt_size(sz)}' for name, sz in sorted(meta['file_sizes'].items())
)
today = dt.date.today().isoformat()
access = (
    'Source: Kaggle competition `talkingdata-adtracking-fraud-detection` '
    '(via `kaggle competitions download`).\n\n'
    f'Local path: `{meta["data_dir"]}`\n\n'
    'File format: CSV.\n\n'
    f'Date inspected: {today}.\n\n'
    'Files on disk:\n' + file_lines
)
print(access)

Source: Kaggle competition `talkingdata-adtracking-fraud-detection` (via `kaggle competitions download`).

Local path: `/home/tsispace/Documents/GitHub/openbotrisk/data/talkingdata-adtracking-fraud-detection`

File format: CSV.

Date inspected: 2026-05-23.

Files on disk:
- `sample_submission.csv` — 186.5 MB
- `test.csv` — 823.3 MB
- `test_supplement.csv` — 2.5 GB
- `train.csv` — 7.0 GB
- `train_sample.csv` — 3.9 MB


In [4]:
tmin = tstats['t_min'].iloc[0]
tmax = tstats['t_max'].iloc[0]
structure = (
    f'- `train.csv`: {meta["row_count"]:,} rows, {len(schema)} columns. One row = one ad click.\n'
    f'- `test.csv`: {meta["test_row_count"] if meta["test_row_count"] is not None else "N/A"} rows (no label).\n'
    f'- Temporal coverage in train: {tmin} to {tmax}.\n'
    '- Single flat table; no join keys between train/test beyond click features.'
)
print(structure)

- `train.csv`: 184,903,890 rows, 8 columns. One row = one ad click.
- `test.csv`: N/A rows (no label).
- Temporal coverage in train: 2017-11-06 14:32:21 to 2017-11-09 16:00:00.
- Single flat table; no join keys between train/test beyond click features.


In [5]:
descriptions = {
    'ip': 'IP address of click (anonymised integer encoding)',
    'app': 'App ID for marketing (encoded)',
    'device': 'Device type ID (encoded, e.g. iphone 6 plus, iphone 7)',
    'os': 'OS version ID (encoded)',
    'channel': 'Channel ID of mobile ad publisher (encoded)',
    'click_time': 'UTC timestamp of click',
    'attributed_time': 'UTC timestamp of app download if click was attributed',
    'is_attributed': 'Target: 1 if app was downloaded after click, else 0',
}
samp_row = sample.iloc[0].to_dict() if len(sample) else {}
schema_rows = ['| column | dtype | example | description |', '|---|---|---|---|']
for _, r in schema.iterrows():
    col = r['column_name']
    schema_rows.append(
        f'| `{col}` | {r["column_type"]} | `{samp_row.get(col, "")}` | {descriptions.get(col, "(anonymised integer encoding)")} |'
    )
schema_md = '\n'.join(schema_rows) + '\n\nAll feature columns (`ip`, `app`, `device`, `os`, `channel`) are anonymised integer IDs with no public mapping.'
print(schema_md[:600])

| column | dtype | example | description |
|---|---|---|---|
| `ip` | BIGINT | `83230` | IP address of click (anonymised integer encoding) |
| `app` | BIGINT | `3` | App ID for marketing (encoded) |
| `device` | BIGINT | `1` | Device type ID (encoded, e.g. iphone 6 plus, iphone 7) |
| `os` | BIGINT | `13` | OS version ID (encoded) |
| `channel` | BIGINT | `379` | Channel ID of mobile ad publisher (encoded) |
| `click_time` | TIMESTAMP | `2017-11-06 14:32:21` | UTC timestamp of click |
| `attributed_time` | TIMESTAMP | `NaT` | UTC timestamp of app download if click was attributed |
| `is_attrib


In [6]:
lbl_sorted = lbl.sort_values('is_attributed').reset_index(drop=True)
total = lbl_sorted['n'].sum()
lines = ['| is_attributed | count | rate |', '|---|---|---|']
for _, r in lbl_sorted.iterrows():
    lines.append(f'| {int(r["is_attributed"])} | {int(r["n"]):,} | {r["n"]/total:.5f} |')
label_md = (
    'Label column: `is_attributed` (1 = click led to an app install, 0 = not).\n\n'
    + '\n'.join(lines) + '\n\n'
    'The label is heavily imbalanced toward zero. `attributed_time` is populated only when '
    '`is_attributed = 1`; it is the time of the attributed install, so the label is time-delayed.'
)
print(label_md)

Label column: `is_attributed` (1 = click led to an app install, 0 = not).

| is_attributed | count | rate |
|---|---|---|
| 0 | 184,447,044 | 0.99753 |
| 1 | 456,846 | 0.00247 |

The label is heavily imbalanced toward zero. `attributed_time` is populated only when `is_attributed = 1`; it is the time of the attributed install, so the label is time-delayed.


In [7]:
n = meta['row_count']
card_lines = ['| column | n_unique | unique_rate | note |', '|---|---|---|---|']
notes = {
    'ip': 'weak actor identifier; shared by NAT/carrier traffic',
    'device': 'device type, not a per-device fingerprint',
    'os': 'OS version code',
    'channel': 'publisher channel',
    'app': 'app being advertised',
    'is_attributed': 'label',
}
for c, k in card.items():
    card_lines.append(f'| `{c}` | {k:,} | {k/n:.6f} | {notes.get(c, "")} |')
ident_md = (
    'Candidate actor/weak identifiers (all are encoded integers; no PII):\n\n'
    + '\n'.join(card_lines) + '\n\n'
    'There is no per-user ID. `ip` is the closest thing to an actor identifier but is a coarse, '
    'NAT-prone signal. No browser, cookie, or session token columns exist.'
)
print(ident_md)

Candidate actor/weak identifiers (all are encoded integers; no PII):

| column | n_unique | unique_rate | note |
|---|---|---|---|
| `app` | 706 | 0.000004 | app being advertised |
| `device` | 3,475 | 0.000019 | device type, not a per-device fingerprint |
| `os` | 800 | 0.000004 | OS version code |
| `channel` | 202 | 0.000001 | publisher channel |
| `is_attributed` | 2 | 0.000000 | label |
| `ip` | 277,396 | 0.001500 | weak actor identifier; shared by NAT/carrier traffic |

There is no per-user ID. `ip` is the closest thing to an actor identifier but is a coarse, NAT-prone signal. No browser, cookie, or session token columns exist.


In [8]:
temporal_md = (
    f'- `click_time`: UTC timestamp at second granularity. Range {tmin} to {tmax}.\n'
    '- `attributed_time`: UTC timestamp at second granularity; populated only on positive labels.\n'
    f'- Record density: clicks span roughly {(tmax - tmin)} of wall-clock time across the train file. '
    'Click density follows diurnal patterns typical of mobile-ad traffic; not plotted here.'
)
print(temporal_md)

- `click_time`: UTC timestamp at second granularity. Range 2017-11-06 14:32:21 to 2017-11-09 16:00:00.
- `attributed_time`: UTC timestamp at second granularity; populated only on positive labels.
- Record density: clicks span roughly 3 days 01:27:39 of wall-clock time across the train file. Click density follows diurnal patterns typical of mobile-ad traffic; not plotted here.


In [9]:
nl = nulls.copy()
nl_lines = ['| column | null_count | null_rate |', '|---|---|---|']
for _, r in nl.iterrows():
    nl_lines.append(f'| `{r["column"]}` | {int(r["null_count"]):,} | {r["null_rate"]:.5f} |')
missing_md = (
    'Per-column null counts (DuckDB single-pass scan):\n\n'
    + '\n'.join(nl_lines) + '\n\n'
    'Only `attributed_time` is meaningfully missing — by design, it is null whenever '
    '`is_attributed = 0`. Feature columns are fully populated.'
)
print(missing_md[:600])

Per-column null counts (DuckDB single-pass scan):

| column | null_count | null_rate |
|---|---|---|
| `ip` | 0 | 0.00000 |
| `app` | 0 | 0.00000 |
| `device` | 0 | 0.00000 |
| `os` | 0 | 0.00000 |
| `channel` | 0 | 0.00000 |
| `click_time` | 0 | 0.00000 |
| `attributed_time` | 184,447,044 | 0.99753 |
| `is_attributed` | 0 | 0.00000 |

Only `attributed_time` is meaningfully missing — by design, it is null whenever `is_attributed = 0`. Feature columns are fully populated.


In [10]:
quirks_md = (
    '- `train.csv` is ~7 GB; loaded via DuckDB streaming. Pandas would not fit comfortably in memory.\n'
    '- All feature columns are anonymised integer IDs — no human-readable meaning, no PII.\n'
    '- `attributed_time` null whenever `is_attributed = 0`; this is by definition, not data loss.\n'
    '- IP cardinality is large but each IP can map to many devices (mobile NAT).\n'
    f'- `test.csv` has {meta["test_row_count"] if meta["test_row_count"] is not None else "N/A"} rows and lacks `is_attributed` / `attributed_time`.'
)
print(quirks_md)

- `train.csv` is ~7 GB; loaded via DuckDB streaming. Pandas would not fit comfortably in memory.
- All feature columns are anonymised integer IDs — no human-readable meaning, no PII.
- `attributed_time` null whenever `is_attributed = 0`; this is by definition, not data loss.
- IP cardinality is large but each IP can map to many devices (mobile NAT).
- `test.csv` has N/A rows and lacks `is_attributed` / `attributed_time`.


In [11]:
repro_md = (
    'Generated by `notebooks/eda/talkingdata-adtracking.ipynb` which calls `openbotrisk.eda.loaders.load_talkingdata_meta`.\n\n'
    'Run with:\n\n'
    '```bash\n'
    'jupyter nbconvert --to notebook --execute --inplace \\\n'
    '  notebooks/eda/talkingdata-adtracking.ipynb \\\n'
    '  --ExecutePreprocessor.timeout=600\n'
    '```\n\n'
    f'Loader runtime on this machine: {elapsed:.1f}s. '
    'No file is fully materialised in pandas; DuckDB performs the row count, null counts, '
    'cardinality, label counts, and time-range queries in a single pass over `train.csv`.'
)
print(repro_md)

Generated by `notebooks/eda/talkingdata-adtracking.ipynb` which calls `openbotrisk.eda.loaders.load_talkingdata_meta`.

Run with:

```bash
jupyter nbconvert --to notebook --execute --inplace \
  notebooks/eda/talkingdata-adtracking.ipynb \
  --ExecutePreprocessor.timeout=600
```

Loader runtime on this machine: 6.2s. No file is fully materialised in pandas; DuckDB performs the row count, null counts, cardinality, label counts, and time-range queries in a single pass over `train.csv`.


In [12]:
stats = {
    'Access': access,
    'Structure': structure,
    'Schema': schema_md,
    'Label': label_md,
    'Identifier inventory': ident_md,
    'Temporal structure': temporal_md,
    'Missing data': missing_md,
    'Quirks and observations': quirks_md,
    'Reproduction': repro_md,
}
out = write_report(stats, 'TalkingData AdTracking Fraud Detection', REPORT_PATH)
print('wrote', out)

wrote /home/tsispace/Documents/GitHub/openbotrisk/working/eda/talkingdata-adtracking.md
